# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/hands_on/DAY_2_HANDS_ON_SESSION_3_planner_thinking.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





#<font color="red" size="10">
<b>HANDS-ON TIME: 15 mins</b>
</font>

# About the Notebook

Hands-on: Planner thinking vs no-thinking + dense vs MoE (Nebius)

1. Run the **planning** step **with** and **without** a thinking/reasoning configuration (via model choice and/or `extra_body`), and compare **latency** and **plan quality**.
2. Run the same workflow using a **dense** planner model and a **smaller MoE** planner model (IDs from your project), and compare behavior.

This notebook uses the **Nebius Token Factory** [OpenAI-compatible inference API](https://docs.tokenfactory.nebius.com/api-reference/introduction) (`chat.completions`). Pick exact model strings from the live catalog: [Token Factory — models](https://tokenfactory.nebius.com/models).

**API key:** create a key in Nebius Studio / Token Factory and export `NEBIUS_API_KEY` (or paste when prompted).

**No mocking** — cells call the real API when you run them.

## Why `extra_body` matters on Nebius

The hosted OpenAPI allows an **`extra_body`** object for **provider-specific** parameters that are not first-class fields on `ChatCompletionRequest` ([create completion schema](https://docs.tokenfactory.nebius.com/api-reference/inference/create-completion)). Some reasoning-capable checkpoints expose toggles (e.g. template kwargs) through that bag—**the exact keys depend on the model** you selected in the catalog.

- **Thinking / reasoning on:** use a **reasoning-oriented model id** (e.g. a DeepSeek-R1 class id from the catalog) and/or set `NEBIUS_PLANNER_EXTRA_BODY_JSON` to the JSON your model’s card documents.
- **Thinking off:** use a **standard instruct** dense id with **no** `extra_body`, or `{}`.

Latency usually **rises** when the model performs more internal deliberation or emits long chain-of-thought before the final JSON. Quality may improve for **decomposition** tasks—measure both wall time and a simple structural score below.

## Dependencies

In [1]:
%pip install -q "langgraph>=0.2" "openai>=1.40" "python-dotenv>=1.0" "pydantic>=2"

## Configuration

| Variable | Purpose |
|----------|---------|
| `NEBIUS_API_KEY` | Bearer token |
| `NEBIUS_BASE_URL` | Default `https://api.tokenfactory.nebius.com/v1` |
| `NEBIUS_PLANNER_MODEL_DENSE` | “Dense” instruct planner (no thinking extras) |
| `NEBIUS_PLANNER_MODEL_MOE` | Smaller MoE instruct planner for A/B |
| `NEBIUS_PLANNER_MODEL_THINKING` | Reasoning-style planner for thinking runs |
| `NEBIUS_EXECUTOR_MODEL` | Cheap model for the second node (fixed across experiments) |
| `NEBIUS_PLANNER_EXTRA_BODY_JSON` | Optional JSON merged into `extra_body` for planner calls |
| `NEBIUS_PLANNER_MAX_TOKENS` | Planner completion budget (default `4096`; thinking models often need more) |

Default model ids match common **`-fast`** listings on [tokenfactory.nebius.com/models](https://tokenfactory.nebius.com/models); override via env if your project uses different strings.

**Note:** Do not force `response_format=json_object` on the planner here—some thinking endpoints return empty `content` with that mode. The notebook merges `content` + reasoning fields and strips `think` / ` ` `</think>`` blocks before parsing.

In [8]:
import os

NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")

if not NEBIUS_API_KEY:
    print("⚠️ NEBIUS_API_KEY not found.")
    NEBIUS_API_KEY = input("Enter your NEBIUS_API_KEY: ").strip()

NEBIUS_API_KEY = NEBIUS_API_KEY.strip().strip('"').strip("'")




In [9]:
from __future__ import annotations

import json
import os
import re
import time
from typing import Any, NotRequired, TypedDict

from dotenv import load_dotenv
from langgraph.graph import END, START, StateGraph
from openai import OpenAI
from pydantic import BaseModel, Field



NEBIUS_BASE_URL = os.getenv("NEBIUS_BASE_URL", "https://api.tokenfactory.nebius.com/v1").strip().rstrip("/")

NEBIUS_PLANNER_MODEL_DENSE = os.getenv(
    "NEBIUS_PLANNER_MODEL_DENSE",
    "meta-llama/Llama-3.3-70B-Instruct-fast",
).strip()
NEBIUS_PLANNER_MODEL_MOE = os.getenv(
    "NEBIUS_PLANNER_MODEL_MOE",
    "Qwen/Qwen3-235B-A22B-Thinking-2507-fast",
).strip()
NEBIUS_PLANNER_MODEL_THINKING = os.getenv(
    "NEBIUS_PLANNER_MODEL_THINKING",
    "Qwen/Qwen3-Next-80B-A3B-Thinking-fast",
).strip()
NEBIUS_EXECUTOR_MODEL = os.getenv(
    "NEBIUS_EXECUTOR_MODEL",
    "meta-llama/Meta-Llama-3.1-8B-Instruct-fast",
).strip()
NEBIUS_PLANNER_MAX_TOKENS = int(os.getenv("NEBIUS_PLANNER_MAX_TOKENS", "4096"))

_extra = os.getenv("NEBIUS_PLANNER_EXTRA_BODY_JSON", "").strip()
NEBIUS_PLANNER_EXTRA_BODY: dict[str, Any] = json.loads(_extra) if _extra else {}

client = OpenAI(base_url=NEBIUS_BASE_URL, api_key=NEBIUS_API_KEY)

print("Base URL:", NEBIUS_BASE_URL)
print("Dense planner:", NEBIUS_PLANNER_MODEL_DENSE)
print("MoE planner:", NEBIUS_PLANNER_MODEL_MOE)
print("Thinking planner:", NEBIUS_PLANNER_MODEL_THINKING)
print("Executor:", NEBIUS_EXECUTOR_MODEL)
print("Planner max_tokens:", NEBIUS_PLANNER_MAX_TOKENS)
print("Planner extra_body:", NEBIUS_PLANNER_EXTRA_BODY or "{}")

Base URL: https://api.tokenfactory.nebius.com/v1
Dense planner: meta-llama/Llama-3.3-70B-Instruct-fast
MoE planner: Qwen/Qwen3-235B-A22B-Thinking-2507-fast
Thinking planner: Qwen/Qwen3-Next-80B-A3B-Thinking-fast
Executor: meta-llama/Meta-Llama-3.1-8B-Instruct-fast
Planner max_tokens: 4096
Planner extra_body: {}


## Plan schema + prompts

The planner should return **JSON** (possibly after a thinking block). The code merges **`message.content` + `reasoning` / `reasoning_content`** when `content` is empty, strips common **Qwen / DeepSeek** think fences, then parses. If needed, a **repair** call uses the **user objective** plus the raw output so `steps` stay real plan steps—not meta-instructions about JSON.

In [10]:
from typing import Any

PLANNER_SYSTEM = """You are a planning component inside a larger workflow.
Given a user objective, output ONLY valid JSON (no markdown fences) with this shape:
{"goal_restated": str, "steps": [str, ...], "checklist": [str, ...], "risks": [str, ...]}
If you use internal reasoning or thinking tags, the final answer must still end with this JSON only.
Constraints:
- steps: 4-8 concrete ordered actions (each one line) for the USER objective.
- checklist: 3-6 verifiable completion checks.
- risks: 2-5 plausible failure modes.
JSON rules (critical): use straight ASCII double quotes only; escape internal " as \\"; no trailing commas;
no raw line breaks inside strings; avoid apostrophes in strings (write cannot not can't).
Do not add keys. Do not wrap in ```."""


class Plan(BaseModel):
    goal_restated: str = Field(..., min_length=8)
    steps: list[str] = Field(..., min_length=4, max_length=10)
    checklist: list[str] = Field(..., min_length=3, max_length=8)
    risks: list[str] = Field(..., min_length=2, max_length=8)


def _strip_json_fence(s: str) -> str:
    s = (s or "").strip()
    if s.startswith("```"):
        lines = s.split("\n")
        if lines:
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        s = "\n".join(lines).strip()
    return s


def _strip_think_tags(s: str) -> str:
    s = s or ""
    s = re.sub(r"`think`[\s\S]*?`think`", "", s, flags=re.I)
    bt = chr(96)
    dr = f"{bt}redacted_reasoning{bt}"
    if dr in s:
        s = s.rsplit(dr, 1)[-1]
    if "</think>" in s:
        s = s.split("</think>", 1)[-1]
    return s.strip()


def _assistant_visible_text(msg: Any) -> str:
    """Some Nebius thinking models leave message.content empty and put text in reasoning fields."""
    chunks: list[str] = []
    c = getattr(msg, "content", None)
    if isinstance(c, str) and c.strip():
        chunks.append(c.strip())
    elif isinstance(c, list):
        for part in c:
            if isinstance(part, dict) and part.get("type") == "text":
                t = (part.get("text") or "").strip()
                if t:
                    chunks.append(t)
            elif isinstance(part, str) and part.strip():
                chunks.append(part.strip())
    for attr in ("reasoning", "reasoning_content"):
        r = getattr(msg, attr, None)
        if r:
            chunks.append(str(r).strip())
    return "\n\n".join(chunks).strip()


def _extract_first_json_object(s: str) -> str:
    s = _strip_think_tags(_strip_json_fence(s))
    i = s.find("{")
    if i == -1:
        return s
    depth = 0
    for j, ch in enumerate(s[i:], start=i):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return s[i : j + 1]
    return s


_TRAILING_COMMA = re.compile(r",(\s*[}\]])")


def _relax_json_text(blob: str) -> str:
    s = blob.strip()
    s = (
        s.replace(chr(0x201C), '"')
        .replace(chr(0x201D), '"')
        .replace(chr(0x2019), "'")
    )
    s = _TRAILING_COMMA.sub(r"\1", s)
    return s


def parse_plan_from_model_text(raw: str) -> Plan:
    """Parse planner output: strict JSON first, then light repair + raw_decode scan."""
    raw = (raw or "").strip()
    if not raw:
        raise ValueError("Empty planner text (no content or reasoning to parse).")
    payload = _extract_first_json_object(raw)
    if not (payload or "").strip():
        raise ValueError("No JSON object found in planner text.")
    errors: list[str] = []
    for attempt, text in enumerate((payload, _relax_json_text(payload))):
        try:
            return Plan.model_validate_json(text)
        except Exception as e:
            errors.append(f"validate_json[{attempt}]: {e}")
        try:
            data = json.loads(text)
            return Plan.model_validate(data)
        except Exception as e:
            errors.append(f"json.loads[{attempt}]: {e}")
    dec = json.JSONDecoder()
    s = _relax_json_text(payload)
    i = 0
    while True:
        j = s.find("{", i)
        if j == -1:
            break
        try:
            obj, end = dec.raw_decode(s, j)
            return Plan.model_validate(obj)
        except Exception as e:
            errors.append(f"raw_decode@{j}: {e}")
            i = j + 1
    raise ValueError("Could not parse Plan JSON. Last errors: " + " | ".join(errors[-6:]))


def plan_quality_score(p: Plan) -> dict[str, Any]:
    """Cheap automatic signals (not ground-truth quality)."""
    steps_lower = [x.strip().lower() for x in p.steps if x.strip()]
    dup = len(steps_lower) - len(set(steps_lower))
    return {
        "n_steps": len(p.steps),
        "n_checklist": len(p.checklist),
        "n_risks": len(p.risks),
        "duplicate_steps": dup,
        "avg_step_len": round(sum(len(s) for s in p.steps) / max(len(p.steps), 1), 1),
    }


REPAIR_SYSTEM = """You output ONLY one JSON object (no markdown).
Keys exactly: goal_restated, steps, checklist, risks.
- goal_restated: one sentence restating the USER objective from the prompt.
- steps: 4-8 strings. Each step must be a real action for that objective (e.g. define metrics, run pilot).
  Do NOT put meta text in steps (no "fix JSON", "remove markdown", or "output only").
- checklist: 3-6 verifiable checks. risks: 2-5 strings.
Use ASCII double quotes and valid JSON escapes."""


def call_planner(
    topic: str,
    *,
    model: str,
    extra_body: dict[str, Any] | None = None,
    temperature: float = 0.2,
    max_tokens: int | None = None,
) -> tuple[Plan, dict[str, Any]]:
    """Returns parsed plan + metadata (latency, usage, raw head)."""
    mt = max_tokens if max_tokens is not None else NEBIUS_PLANNER_MAX_TOKENS
    body = {k: v for k, v in dict(extra_body or {}).items() if v is not None}
    t0 = time.perf_counter()
    create_kw: dict[str, Any] = {
        "model": model,
        "temperature": temperature,
        "max_tokens": mt,
        "messages": [
            {"role": "system", "content": PLANNER_SYSTEM},
            {"role": "user", "content": topic},
        ],
    }
    if body:
        create_kw["extra_body"] = body
    resp = client.chat.completions.create(**create_kw)
    msg = resp.choices[0].message
    raw = _assistant_visible_text(msg)
    reasoning = getattr(msg, "reasoning", None) or getattr(msg, "reasoning_content", None)
    meta: dict[str, Any] = {"json_repair_used": False}
    try:
        plan = parse_plan_from_model_text(raw)
    except ValueError:
        blob = raw if raw.strip() else str(msg)[:12000]
        fix_user = (
            f"USER OBJECTIVE:\n{topic}\n\n"
            f"PLANNER MODEL OUTPUT (may be broken or mixed with thinking):\n{blob[:12000]}"
        )
        fix = client.chat.completions.create(
            model=NEBIUS_EXECUTOR_MODEL,
            temperature=0,
            max_tokens=2048,
            messages=[
                {"role": "system", "content": REPAIR_SYSTEM},
                {"role": "user", "content": fix_user},
            ],
        )
        fix_msg = fix.choices[0].message
        raw2 = _assistant_visible_text(fix_msg)
        try:
            plan = parse_plan_from_model_text(raw2)
        except ValueError as err2:
            hint = (raw[:500] + "…") if raw else "(empty combined content)"
            raise ValueError(
                "Planner output was not valid JSON even after repair pass. "
                f"Combined planner text (head): {hint}"
            ) from err2
        meta["json_repair_used"] = True
    wall_s = time.perf_counter() - t0
    usage = None
    if getattr(resp, "usage", None) is not None:
        u = resp.usage
        usage = u.model_dump() if hasattr(u, "model_dump") else dict(u)
    meta.update(
        {
            "wall_s": round(wall_s, 4),
            "usage": usage,
            "raw_head": raw[:400] + ("…" if len(raw) > 400 else ""),
            "reasoning_head": (
                (str(reasoning)[:400] + "…") if reasoning else None
            ),
        }
    )
    return plan, meta

## LangGraph workflow: `plan` → `execute`

The **executor** is deliberately simple: it turns the structured plan into a short Markdown summary using a **fixed** small model so differences mostly reflect the **planner**.

In [11]:
class WorkflowState(TypedDict):
    topic: str
    planner_model: str
    planner_label: str
    planner_extra_body: NotRequired[dict[str, Any]]
    plan: NotRequired[dict]
    plan_meta: NotRequired[dict]
    execution: NotRequired[str]
    execute_meta: NotRequired[dict]
    timings: NotRequired[dict[str, float]]


def node_plan(state: WorkflowState) -> dict:
    extra = dict(state.get("planner_extra_body") or {})
    plan, meta = call_planner(
        state["topic"],
        model=state["planner_model"],
        extra_body=extra,
    )
    tim = dict(state.get("timings") or {})
    tim["plan_llm_s"] = float(meta["wall_s"])
    return {
        "plan": plan.model_dump(),
        "plan_meta": meta,
        "timings": tim,
    }


def node_execute(state: WorkflowState) -> dict:
    t0 = time.perf_counter()
    user = (
        f"Planner label: {state['planner_label']}\n"
        f"Objective:\n{state['topic']}\n\n"
        f"Structured plan (JSON):\n{json.dumps(state['plan'], ensure_ascii=False)}\n\n"
        "Write a concise Markdown brief: ## Summary, ## Recommended order, ## Watchouts. "
        "Do not change the plan JSON; interpret it."
    )
    resp = client.chat.completions.create(
        model=NEBIUS_EXECUTOR_MODEL,
        temperature=0.3,
        max_tokens=900,
        messages=[{"role": "user", "content": user}],
    )
    wall_s = time.perf_counter() - t0
    text = (resp.choices[0].message.content or "").strip()
    usage = None
    if getattr(resp, "usage", None) is not None:
        u = resp.usage
        usage = u.model_dump() if hasattr(u, "model_dump") else dict(u)
    tim = dict(state.get("timings") or {})
    tim["execute_llm_s"] = round(wall_s, 4)
    return {
        "execution": text,
        "execute_meta": {"wall_s": tim["execute_llm_s"], "usage": usage},
        "timings": tim,
    }


g = StateGraph(WorkflowState)
g.add_node("plan", node_plan)
g.add_node("execute", node_execute)
g.add_edge(START, "plan")
g.add_edge("plan", "execute")
g.add_edge("execute", END)
app = g.compile()

try:
    print(app.get_graph().draw_mermaid())
except Exception as exc:
    print("(Mermaid skipped)", exc)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan(plan)
	execute(execute)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan;
	plan --> execute;
	execute --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Experiment A — same dense planner: **no extra** vs **`extra_body`** vs **reasoning model**

1. **No-thinking baseline** — dense instruct model, empty `extra_body`.
2. **Same model + catalog extras** — merges `NEBIUS_PLANNER_EXTRA_BODY_JSON` (if your model documents one).
3. **Thinking-style model** — swap to `NEBIUS_PLANNER_MODEL_THINKING` (typical higher latency; may improve decomposition).

In [12]:
TOPIC = os.getenv(
    "NEBIUS_DEMO_TOPIC",
    "Design a short evaluation plan for a customer-support agent before production rollout.",
)


def run_labeled(label: str, planner_model: str, extra: dict[str, Any] | None) -> dict:
    t_wall0 = time.perf_counter()
    out = app.invoke(
        {
            "topic": TOPIC,
            "planner_model": planner_model,
            "planner_label": label,
            "planner_extra_body": extra or {},
        }
    )
    wall = time.perf_counter() - t_wall0
    plan = Plan.model_validate(out["plan"])
    score = plan_quality_score(plan)
    return {
        "label": label,
        "planner_model": planner_model,
        "extra_body": extra or {},
        "wall_total_s": round(wall, 4),
        "timings": out.get("timings") or {},
        "quality": score,
        "plan": out["plan"],
        "plan_meta": out.get("plan_meta") or {},
        "execution_excerpt": (out.get("execution") or "")[:1500],
    }


rows_a = [run_labeled("dense_no_extra", NEBIUS_PLANNER_MODEL_DENSE, None)]
if NEBIUS_PLANNER_EXTRA_BODY:
    rows_a.append(
        run_labeled("dense_with_extra_body", NEBIUS_PLANNER_MODEL_DENSE, NEBIUS_PLANNER_EXTRA_BODY)
    )
else:
    print("(Skip dense_with_extra_body: set NEBIUS_PLANNER_EXTRA_BODY_JSON to a JSON object.)")
rows_a.append(run_labeled("thinking_model", NEBIUS_PLANNER_MODEL_THINKING, None))

print("Topic:", TOPIC)
for r in rows_a:
    print("\n===", r["label"], "===")
    print("planner:", r["planner_model"], "| extra:", r["extra_body"] or "{}")
    print("wall_total_s:", r["wall_total_s"], "| node timings:", r["timings"])
    if r.get("plan_meta", {}).get("json_repair_used"):
        print("(Planner JSON needed a repair pass via executor model — adds latency.)")
    print("quality:", r["quality"])
    if r["plan_meta"].get("reasoning_head"):
        print("reasoning_head:", r["plan_meta"]["reasoning_head"])
    print("execution_excerpt:\n", r["execution_excerpt"])

(Skip dense_with_extra_body: set NEBIUS_PLANNER_EXTRA_BODY_JSON to a JSON object.)
Topic: Design a short evaluation plan for a customer-support agent before production rollout.

=== dense_no_extra ===
planner: meta-llama/Llama-3.3-70B-Instruct-fast | extra: {}
wall_total_s: 11.3592 | node timings: {'plan_llm_s': 10.0285, 'execute_llm_s': 1.3286}
(Planner JSON needed a repair pass via executor model — adds latency.)
quality: {'n_steps': 7, 'n_checklist': 5, 'n_risks': 5, 'duplicate_steps': 0, 'avg_step_len': 71.4}
execution_excerpt:
 ## Summary
This evaluation plan aims to assess the performance of customer-support agents before a production rollout. It involves developing a comprehensive evaluation framework, conducting mock customer interactions, and assessing technical skills. The plan also includes providing feedback and coaching to agents and refining the evaluation framework as needed.

## Recommended Order
1. Develop a comprehensive evaluation framework with key performance indic

## Experiment B — **dense** vs **MoE** planner (same `extra_body` policy)

Hold the executor fixed. Compare planner models from the catalog: a **dense** instruct vs a **smaller MoE** instruct. If the MoE id is unavailable in your project, change `NEBIUS_PLANNER_MODEL_MOE`.

In [13]:
rows_b = [
    run_labeled("planner_dense", NEBIUS_PLANNER_MODEL_DENSE, None),
    run_labeled("planner_moe", NEBIUS_PLANNER_MODEL_MOE, None),
]

for r in rows_b:
    print("\n===", r["label"], "===")
    print("planner:", r["planner_model"])
    print("wall_total_s:", r["wall_total_s"], "| plan node:", r["timings"].get("plan_llm_s"))
    if r.get("plan_meta", {}).get("json_repair_used"):
        print("(JSON repair pass used.)")
    print("quality:", r["quality"])

print("\n--- Side-by-side ---")
print(
    json.dumps(
        [
            {
                "label": r["label"],
                "model": r["planner_model"],
                "wall_total_s": r["wall_total_s"],
                "plan_s": r["timings"].get("plan_llm_s"),
                "quality": r["quality"],
            }
            for r in rows_b
        ],
        indent=2,
    )
)


=== planner_dense ===
planner: meta-llama/Llama-3.3-70B-Instruct-fast
wall_total_s: 8.8141 | plan node: 7.2448
(JSON repair pass used.)
quality: {'n_steps': 7, 'n_checklist': 5, 'n_risks': 5, 'duplicate_steps': 0, 'avg_step_len': 79.3}

=== planner_moe ===
planner: Qwen/Qwen3-235B-A22B-Thinking-2507-fast
wall_total_s: 6.4822 | plan node: 5.3012
quality: {'n_steps': 6, 'n_checklist': 4, 'n_risks': 3, 'duplicate_steps': 0, 'avg_step_len': 101.3}

--- Side-by-side ---
[
  {
    "label": "planner_dense",
    "model": "meta-llama/Llama-3.3-70B-Instruct-fast",
    "wall_total_s": 8.8141,
    "plan_s": 7.2448,
    "quality": {
      "n_steps": 7,
      "n_checklist": 5,
      "n_risks": 5,
      "duplicate_steps": 0,
      "avg_step_len": 79.3
    }
  },
  {
    "label": "planner_moe",
    "model": "Qwen/Qwen3-235B-A22B-Thinking-2507-fast",
    "wall_total_s": 6.4822,
    "plan_s": 5.3012,
    "quality": {
      "n_steps": 6,
      "n_checklist": 4,
      "n_risks": 3,
      "duplicate_steps